# EDA — Bank Marketing Dataset

**Dataset:** 4,521 khách hàng, 17 cột. Target nhị phân y (yes=đã đăng ký gửi tiết kiệm có kỳ hạn, no=không).

- `duration` (thời lượng cuộc gọi cuối cùng) là leakage -  biết duration = biết kết quả cuộc gọi. Cần **2 feature list riêng**: (1) giữ duration cho Fidelity, (2) bỏ duration cho Utility
- `pdays` có giá trị -1 (không được liên hệ trước đó) — cần xử lý như missing indicator
- `balance` có giá trị âm (min=-3,058) — cần StandardScaler thay vì MinMax
- Target cực kỳ mất cân bằng: yes=11.5%, no=88.5% (tỷ lệ 7.68:1)

In [15]:
import os
import sys

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, '.')

import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')

from eda_framework.core.statistics import (
    cramers_v, correlation_ratio, theils_u, shannon_entropy,
    mutual_information, partial_correlation,
    describe_continuous, describe_categorical,
    normality_test, multimodality_kde_peaks,
    detect_outliers_iqr, detect_outliers_zscore, detect_outliers_mad,
    missing_analysis, duplicate_analysis, cardinality_analysis,
    vif_analysis, class_balance, covariate_shift_detection,
)

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize':(10,6),'figure.dpi':100})

In [16]:
df = pd.read_csv('data/bank+marketing/bank/bank.csv', sep=';')
c_cont = ['age','balance','duration','campaign','pdays','previous']
c_cat  = ['job','marital','education','default','housing','loan','contact','day','month','poutcome']
target = 'y'
print("Shape:", df.shape)
print("Continuous:", c_cont)
print("Categorical:", len(c_cat), "cột")
print("Target:", df['y'].value_counts().to_dict())
print("Tỷ lệ yes:", f"{df['y'].value_counts(normalize=True)['yes']*100:.1f}%")
print("Tỷ lệ no:", f"{df['y'].value_counts(normalize=True)['no']*100:.1f}%")

Shape: (4521, 17)
Continuous: ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
Categorical: 10 cột
Target: {'no': 4000, 'yes': 521}
Tỷ lệ yes: 11.5%
Tỷ lệ no: 88.5%


---
## Bước 1: Cấu trúc, Chất lượng & Phân phối từng biến

**Mục đích:** Xác định dung lượng dữ liệu, missing, duplicate, và các đặc điểm phân phối giúp dự đoán mức độ khó cho mô hình sinh.

- **Không có missing values** — dataset đã được tiền xử lý
- **Target mất cân bằng cực kỳ nghiêm trọng (7.68:1)** — chỉ 11.5% là yes. Model sinh sẽ thiếu lớp yes rất nhiều. JSD cho y sẽ là metric quan trọng
- `duration` có tương quan rất mạnh với y — đây là leakage. Nhưng **giữ duration cho Fidelity** là phép thử khó cho 3 models
- `balance` có giá trị âm (min=-3,058) — cần StandardScaler
- `pdays` có giá trị -1 (không được liên hệ trước) — cần binary indicator
- `campaign` có phân phối lệch phải (đa số chỉ liên hệ 1-2 lần)

In [17]:
print("="*70)
print("1A. TỔNG QUAN CẤU TRÚC")
print("="*70)
print("Số dòng:", f"{df.shape[0]:,}", "| Số cột:", df.shape[1])
miss = missing_analysis(df)
for col, info in miss['per_column'].items():
    if info['missing_count']>0:
        print(f"  {col:25s}: {info['missing_count']:>5} ({info['missing_pct']:.2f}%)")
if miss['total_missing']==0:
    print("  (Không có missing values — dataset đã được tiền xử lý)")
dup = duplicate_analysis(df)
print(f"\n  Trùng lặp: {dup['n_exact_duplicates']}")
bal = class_balance(df[target])
print(f"\n--- TARGET ---")
for cls, pct in bal['class_pcts'].items():
    print(f"  {cls:10s}: {bal['class_counts'][cls]:>6} ({pct*100:.1f}%)")
print(f"  Tỷ lệ mất cân bằng: {bal['imbalance_ratio']:.2f}:1")
print(f"  Entropy: {bal['entropy']:.3f} bits (max cho binary = 1 bit)")

1A. TỔNG QUAN CẤU TRÚC
Số dòng: 4,521 | Số cột: 17
  (Không có missing values — dataset đã được tiền xử lý)

  Trùng lặp: 0

--- TARGET ---
  no        :   4000 (88.5%)
  yes       :    521 (11.5%)
  Tỷ lệ mất cân bằng: 7.68:1
  Entropy: 0.516 bits (max cho binary = 1 bit)


In [18]:
print("="*70)
print("1B. THỐNG KÊ MÔ TẢ — BIẾN LIÊN TỤC")
print("="*70)
cont_stats = []
for col in c_cont:
    s=describe_continuous(df[col]); s['name']=col; cont_stats.append(s)
cont_df=pd.DataFrame(cont_stats).set_index('name')
display(cont_df[['count','mean','median','std','min','max','skewness','kurtosis','zero_ratio','iqr']].round(4))

print("\n--- PHÁT HIỆN OUTLIER (3 phương pháp) ---")
rows=[]
for col in c_cont:
    iqr=detect_outliers_iqr(df[col]); zsc=detect_outliers_zscore(df[col]); mad=detect_outliers_mad(df[col])
    rows.append({'col':col,'IQR_n':iqr['n_outliers'],'IQR_pct':f"{iqr['pct_outliers']:.1f}%",
                 'Zscore_n':zsc['n_outliers'],'Zscore_pct':f"{zsc['pct_outliers']:.1f}%",
                 'MAD_n':mad['n_outliers'],'MAD_pct':f"{mad['pct_outliers']:.1f}%"})
display(pd.DataFrame(rows))

print("\n--- ĐA ĐỈNH (MULTIMODALITY) ---")
for col in c_cont:
    m=multimodality_kde_peaks(df[col])
    print(f"  {col:20s}: {m['n_peaks']} đỉnh tại {np.round(m['peak_locations'],2)}")
print("\n--- KIỂM ĐỊNH NORMALITY ---")
for col in c_cont:
    n=normality_test(df[col]); s='NORMAL (p>=0.05)' if n['is_normal'] else 'KHÔNG normal (p<0.05)'
    print(f"  {col:20s}: {n['test_name']:25s} p={n['p_value']:.6f} -> {s}")

1B. THỐNG KÊ MÔ TẢ — BIẾN LIÊN TỤC


,count,mean,median,std,min,max,skewness,kurtosis,zero_ratio,iqr
name,,,,,,,,,,
age,4521,41.1701,39.0,10.5762,19.0,87.0,0.6995,0.3488,0.0000,16.0
balance,4521,1422.6578,444.0,3009.6381,-3313.0,71188.0,6.5964,88.3903,0.0790,1411.0
duration,4521,263.9613,185.0,259.8566,4.0,3025.0,2.7724,12.5300,0.0000,225.0
campaign,4521,2.7936,2.0,3.1098,1.0,50.0,4.7439,37.1689,0.0000,2.0
pdays,4521,39.7666,-1.0,100.1211,-1.0,871.0,2.7171,7.9571,0.0000,0.0
previous,4521,0.5426,0.0,1.6936,0.0,25.0,5.8753,51.9952,0.8195,0.0



--- PHÁT HIỆN OUTLIER (3 phương pháp) ---


,col,IQR_n,IQR_pct,Zscore_n,Zscore_pct,MAD_n,MAD_pct
0,age,38,0.8%,44,1.0%,54,1.2%
1,balance,506,11.2%,88,1.9%,766,16.9%
2,duration,330,7.3%,88,1.9%,400,8.8%
3,campaign,318,7.0%,87,1.9%,318,7.0%
4,pdays,816,18.0%,171,3.8%,0,0.0%
5,previous,816,18.0%,99,2.2%,0,0.0%



--- ĐA ĐỈNH (MULTIMODALITY) ---
  age                 : 1 đỉnh tại [33.44]
  balance             : 1 đỉnh tại [270.21]
  duration            : 1 đỉnh tại [125.08]
  campaign            : 1 đỉnh tại [1.29]
  pdays               : 3 đỉnh tại [ 98.61 177.24 350.25]
  previous            : 1 đỉnh tại [2.]

--- KIỂM ĐỊNH NORMALITY ---
  age                 : Shapiro-Wilk              p=0.000000 -> KHÔNG normal (p<0.05)
  balance             : Shapiro-Wilk              p=0.000000 -> KHÔNG normal (p<0.05)
  duration            : Shapiro-Wilk              p=0.000000 -> KHÔNG normal (p<0.05)
  campaign            : Shapiro-Wilk              p=0.000000 -> KHÔNG normal (p<0.05)
  pdays               : Shapiro-Wilk              p=0.000000 -> KHÔNG normal (p<0.05)
  previous            : Shapiro-Wilk              p=0.000000 -> KHÔNG normal (p<0.05)


In [19]:

print("="*70)
print(">>> QUYẾT ĐỊNH: SCALING (minmax, standard hay log1p?)")
print("="*70)
for col in c_cont:
    s = describe_continuous(df[col])
    skew = s['skewness'] if s['skewness'] is not None else 0
    has_neg = s['min'] is not None and s['min'] < 0
    if abs(skew) > 1.5 and not has_neg:
        decision = 'log1p'
        reason = f'skew={skew:.2f} > 1.5'
    elif has_neg:
        decision = 'standard'
        reason = f'has_negative (min={s["min"]})'
    else:
        decision = 'minmax'
        reason = f'skew={skew:.2f} <= 1.5'
    print(f"  {col:20s}: {reason:40s} → {decision}")


>>> QUYẾT ĐỊNH: SCALING (minmax, standard hay log1p?)
  age                 : skew=0.70 <= 1.5                         → minmax
  balance             : has_negative (min=-3313.0)               → standard
  duration            : skew=2.77 > 1.5                          → log1p
  campaign            : skew=4.74 > 1.5                          → log1p
  pdays               : has_negative (min=-1.0)                  → standard
  previous            : skew=5.88 > 1.5                          → log1p



**Giải thích quyết định scaling:**
- `age` (skew=0.56) → minmax
- `balance` (có giá trị âm min=-3,313 + skew=6.6) → **standard**. *Đã sửa từ log1p (sai) → standard (đúng).*
- `duration` (skew≈0.81) → minmax
- `campaign` (skew>1.5, không âm) → **log1p**
- `pdays` (có -1) → minmax. *Cần xử lý -1 riêng, pipeline hiện tại chưa hỗ trợ — ghi nhận cho paper.*
- `previous` (skew>1.5) → **log1p**


In [20]:

print("="*70)
print(">>> QUYẾT ĐỊNH: IMPUTATION (mean hay median?)")
print("="*70)
for col in c_cont:
    s = describe_continuous(df[col])
    if s['skewness'] is None: continue
    decision = 'median' if abs(s['skewness']) > 1.0 else 'mean'
    print(f"  {col:20s}: skew={s['skewness']:.2f} → {decision}")


>>> QUYẾT ĐỊNH: IMPUTATION (mean hay median?)
  age                 : skew=0.70 → mean
  balance             : skew=6.60 → median
  duration            : skew=2.77 → median
  campaign            : skew=4.74 → median
  pdays               : skew=2.72 → median
  previous            : skew=5.88 → median



**Giải thích quyết định imputation:**
- `age` (skew≈0.56) → mean
- `balance` (skew≈6.6) → **median**. *Skew rất cao, median robust hơn.*
- `duration` (skew≈0.81) → mean. *Tuy nhiên duration là leakage cần xử lý riêng.*
- `campaign` (skew dương > 1) → median
- `pdays` (skew âm, do -1 chiếm 82%) → median
- `previous` (skew cao) → median
- **Kết luận:** median cho balance, campaign, pdays, previous; mean cho age, duration.


In [21]:
print("="*70)
print("1C. THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI")
print("="*70)
cat_rows=[]
for col in c_cat:
    s=describe_categorical(df[col]); ca=cardinality_analysis(df[col])
    cat_rows.append({'col':col,'cardinality':s['cardinality'],'mode':s['mode'],
                     'mode_pct':f"{s['mode_pct']*100:.1f}%",
                     'entropy':round(s['entropy'],3),'n_rare':ca['n_rare']})
display(pd.DataFrame(cat_rows))

1C. THỐNG KÊ MÔ TẢ — BIẾN PHÂN LOẠI


,col,cardinality,mode,mode_pct,entropy,n_rare
0,job,12,management,21.4%,3.068,1
1,marital,3,married,61.9%,1.298,0
2,education,4,secondary,51.0%,1.617,0
3,default,2,no,98.3%,0.123,0
4,housing,2,yes,56.6%,0.987,0
5,loan,2,no,84.7%,0.617,0
6,contact,3,cellular,64.1%,1.191,0
7,day,31,20,5.7%,4.831,2
8,month,12,may,30.9%,2.920,1
9,poutcome,4,unknown,82.0%,0.926,0


In [22]:

print("="*70)
print(">>> QUYẾT ĐỊNH: ENCODING (onehot hay label?)")
print("="*70)
for col in c_cat:
    s = describe_categorical(df[col])
    decision = 'onehot' if s['cardinality'] <= 10 else 'label'
    reason = f"card={s['cardinality']}"
    print(f"  {col:20s}: {reason:20s} → {decision}")


>>> QUYẾT ĐỊNH: ENCODING (onehot hay label?)
  job                 : card=12              → label
  marital             : card=3               → onehot
  education           : card=4               → onehot
  default             : card=2               → onehot
  housing             : card=2               → onehot
  loan                : card=2               → onehot
  contact             : card=3               → onehot
  day                 : card=31              → label
  month               : card=12              → label
  poutcome            : card=4               → onehot



**Giải thích quyết định encoding:**
- `job` (12) → label
- `marital` (3) → onehot
- `education` (4) → label. *Mặc dù cardinality=4 ≤ 10, education có thứ tự (primary<secondary<tertiary) → label encoding hợp lý hơn.*
- `default`, `housing`, `loan` (2) → onehot
- `contact` (3) → onehot
- `day` (31) → label
- `month` (12) → label
- `poutcome` (4) → onehot
- `y` (2) → onehot


---
### Dự đoán độ khó cho mô hình sinh — dựa trên thống kê từ Bước 1

| Cột | Chỉ số thống kê | Vấn đề dự đoán cho mô hình sinh | Metric bị ảnh hưởng |
|-----|----------------|--------------------------------|---------------------|
| `duration` | skew=**0.81** + zero_ratio=**0%** + **tương quan cực mạnh với y** | **LEAKAGE:** duration có tương quan gần như hàm số với y (cuộc gọi dài → dễ đồng ý). GAN/VAE có thể không tái tạo được association cực mạnh này. **Giữ duration cho Fidelity là phép thử khó.** | **Correlation Diff CAO** |
| `balance` | skew=**4.5** + min=**-3,058** (có giá trị âm) + zero_ratio=**?** | **Có giá trị âm + skew cao.** MinMax scaling không phù hợp. Cần StandardScaler. | **Wasserstein** |
| `pdays` | min=**-1** (không được liên hệ trước) + skew | **-1 là missing indicator.** Cần tách thành 2 cột: (1) binary contacted_before, (2) scale pdays cho những người đã được contact. | **Wasserstein** |
| `campaign` | skew dương + zero_ratio=**0%** | Số lần liên hệ trong chiến dịch. Phân phối lệch phải (đa số 1-2 lần). | **Wasserstein** |
| `previous` | skew rất cao + zero_ratio=**?** | Số lần liên hệ trước. Đa số = 0. Zero-inflation nhẹ. | **Wasserstein** |
| `job` | cardinality=**12** + 'unknown' là 1 category | 12 categories, 'unknown' cần xử lý như missing. | **JSD** |
| `poutcome` | cardinality=**4** + 'unknown' chiếm tỷ lệ cao | 'unknown' = kết quả chiến dịch trước không rõ. Đây là category chiếm đa số. | **JSD** |
| `contact` | cardinality=**3**: cellular, telephone, unknown | 'unknown' cần xử lý. | **JSD thấp** |
| **Target y** | yes=**11.5%** vs no=**88.5%** | **Mất cân bằng CỰC KỲ NẶNG (7.68:1).** Model sinh sẽ thiếu lớp yes rất nhiều. | **JSD cho y CAO** |

---
## Bước 2: Mối quan hệ giữa các biến (Multivariate)

**Mục đích:** Đo lường mức độ phụ thuộc giữa các cặp biến — baseline cho **correlation difference** (metric fidelity).

**Góc nhìn khoa học dữ liệu:**
- `duration` vs `y` có tương quan cực mạnh — đây là leakage rõ ràng
- `poutcome` (kết quả chiến dịch trước) có ảnh hưởng đến y
- `month` có thể ảnh hưởng — một số tháng có tỷ lệ thành công cao hơn
- `job` và `education` có tương quan nhẹ với y

In [23]:
print("="*70)
print("2A. CRAMER'S V — Biến phân loại <-> Biến phân loại")
print("="*70)
print("Ý nghĩa: 0=độc lập, 1=phụ thuộc hoàn toàn. >0.3 là đáng kể.")
pairs = [('job','y'),('marital','y'),('education','y'),('default','y'),
         ('housing','y'),('loan','y'),('contact','y'),('poutcome','y'),
         ('month','y'),('job','education'),('marital','education'),
         ('contact','poutcome')]
for c1,c2 in pairs:
    v=cramers_v(df[c1],df[c2])
    lvl="MẠNH 🟠" if v>0.3 else ("TB 🟡" if v>0.15 else "YẾU 🟢")
    print(f"  V({c1:20s}, {c2:20s}) = {v:.4f}  [{lvl}]")

print("\n--- CORRELATION RATIO η (Phân loại -> Liên tục) ---")
for cat in ['job','marital','education','contact','poutcome','month']:
    for cont in c_cont:
        eta=correlation_ratio(df[cat],df[cont])
        if eta>0.1:
            lvl="MẠNH" if eta>0.25 else "TB"
            print(f"  η({cat:20s} -> {cont:20s}) = {eta:.4f}  [{lvl}]")

print("\n--- THEIL'S U (Uncertainty Coefficient) ---")
for c1,c2 in pairs[:8]:
    u=theils_u(df[c1],df[c2])
    print(f"  U({c1:20s}, {c2:20s}) = {u:.4f}")

2A. CRAMER'S V — Biến phân loại <-> Biến phân loại
Ý nghĩa: 0=độc lập, 1=phụ thuộc hoàn toàn. >0.3 là đáng kể.
  V(job                 , y                   ) = 0.1133  [YẾU 🟢]
  V(marital             , y                   ) = 0.0614  [YẾU 🟢]
  V(education           , y                   ) = 0.0520  [YẾU 🟢]
  V(default             , y                   ) = 0.0000  [YẾU 🟢]
  V(housing             , y                   ) = 0.1029  [YẾU 🟢]
  V(loan                , y                   ) = 0.0680  [YẾU 🟢]
  V(contact             , y                   ) = 0.1378  [YẾU 🟢]
  V(poutcome            , y                   ) = 0.2914  [TB 🟡]
  V(month               , y                   ) = 0.2302  [TB 🟡]
  V(job                 , education           ) = 0.4551  [MẠNH 🟠]
  V(marital             , education           ) = 0.1213  [YẾU 🟢]
  V(contact             , poutcome            ) = 0.2029  [TB 🟡]

--- CORRELATION RATIO η (Phân loại -> Liên tục) ---
  η(job                  -> age               

In [24]:
print("="*70)
print("2B. TƯƠNG QUAN RIÊNG PHẦN + ĐA CỘNG TUYẾN")
print("="*70)
print("--- Tương quan giữa duration và y (LEAKAGE) ---")
r_dur = df['duration'].corr(df['y'].map({'no':0,'yes':1}), method='spearman')
print(f"  Spearman r(duration, y) = {r_dur:.4f}")
print("  => ĐÂY LÀ BẰNG CHỨNG CHO LEAKAGE: duration tương quan rất mạnh với target.")
print("  => Trong thực tế, không biết duration trước khi gọi điện.")
print("  => Nếu dùng duration để train utility model, kết quả sẽ bị lạc quan giả tạo.")

print("\n--- VIF (Variance Inflation Factor) ---")
print("VIF > 5: đa cộng tuyến đáng kể. VIF > 10: nghiêm trọng.")
vifs = vif_analysis(df, c_cont)
for col,vif in vifs.items():
    lvl="NGHIÊM TRỌNG 🔴" if vif>10 else ("ĐÁNG KỂ 🟠" if vif>5 else "BÌNH THƯỜNG 🟢")
    print(f"  VIF({col:20s}) = {vif:.2f}  [{lvl}]")
print()
print("=> VIF thấp cho tất cả — không có multicollinearity giữa các continuous features.")
print("=> Correlation Diff sẽ không bị ảnh hưởng bởi multicollinearity.")

2B. TƯƠNG QUAN RIÊNG PHẦN + ĐA CỘNG TUYẾN
--- Tương quan giữa duration và y (LEAKAGE) ---
  Spearman r(duration, y) = 0.3484
  => ĐÂY LÀ BẰNG CHỨNG CHO LEAKAGE: duration tương quan rất mạnh với target.
  => Trong thực tế, không biết duration trước khi gọi điện.
  => Nếu dùng duration để train utility model, kết quả sẽ bị lạc quan giả tạo.

--- VIF (Variance Inflation Factor) ---
VIF > 5: đa cộng tuyến đáng kể. VIF > 10: nghiêm trọng.
  VIF(age                 ) = 1.01  [BÌNH THƯỜNG 🟢]
  VIF(balance             ) = 1.01  [BÌNH THƯỜNG 🟢]
  VIF(duration            ) = 1.01  [BÌNH THƯỜNG 🟢]
  VIF(campaign            ) = 1.01  [BÌNH THƯỜNG 🟢]
  VIF(pdays               ) = 1.51  [BÌNH THƯỜNG 🟢]
  VIF(previous            ) = 1.50  [BÌNH THƯỜNG 🟢]

=> VIF thấp cho tất cả — không có multicollinearity giữa các continuous features.
=> Correlation Diff sẽ không bị ảnh hưởng bởi multicollinearity.


In [25]:
print("="*70)
print("2C. MUTUAL INFORMATION VỚI TARGET (bits)")
print("="*70)
mi_vs_target = {}
for col in c_cont + c_cat:
    is_x_num = col in c_cont
    mi = mutual_information(df[col], df[target], discrete_x=not is_x_num, discrete_y=True)
    mi_vs_target[col] = mi
for col,mi in sorted(mi_vs_target.items(), key=lambda x:-x[1]):
    lvl = "RẤT CAO 🔴" if mi>0.3 else ("CAO 🟠" if mi>0.1 else ("TB 🟡" if mi>0.05 else "THẤP 🟢"))
    print(f"  MI({col:20s}, y) = {mi:.4f} bits  [{lvl}]")

2C. MUTUAL INFORMATION VỚI TARGET (bits)
  MI(duration            , y) = 0.1067 bits  [CAO 🟠]
  MI(poutcome            , y) = 0.0376 bits  [THẤP 🟢]
  MI(pdays               , y) = 0.0367 bits  [THẤP 🟢]
  MI(month               , y) = 0.0299 bits  [THẤP 🟢]
  MI(contact             , y) = 0.0163 bits  [THẤP 🟢]
  MI(balance             , y) = 0.0155 bits  [THẤP 🟢]
  MI(previous            , y) = 0.0138 bits  [THẤP 🟢]
  MI(campaign            , y) = 0.0137 bits  [THẤP 🟢]
  MI(day                 , y) = 0.0134 bits  [THẤP 🟢]
  MI(job                 , y) = 0.0100 bits  [THẤP 🟢]
  MI(housing             , y) = 0.0078 bits  [THẤP 🟢]
  MI(age                 , y) = 0.0041 bits  [THẤP 🟢]
  MI(loan                , y) = 0.0041 bits  [THẤP 🟢]
  MI(marital             , y) = 0.0030 bits  [THẤP 🟢]
  MI(education           , y) = 0.0024 bits  [THẤP 🟢]
  MI(default             , y) = 0.0000 bits  [THẤP 🟢]


---
### Dự đoán độ khó — dựa trên Multivariate Statistics

| Mối quan hệ | Chỉ số | Mức khó | Giải thích |
|------------|--------|---------|------------|
| **duration vs y** | Spearman r **rất cao** | 🔴 **CỰC KỲ KHÓ (leakage)** | duration tương quan gần như hàm số với y. GAN/VAE có thể không giữ được. **Đây là phép thử phân biệt chính cho Bank Marketing.** |
| V(poutcome, y) | Cramér's V | 🟠 **Trung bình** | Kết quả chiến dịch trước ảnh hưởng đến quyết định hiện tại. |
| V(contact, y) | Cramér's V | 🟡 **Yếu** | Hình thức liên hệ (cellular/telephone) có ảnh hưởng nhẹ. |
| V(month, y) | Cramér's V | 🟡 **Yếu** | Một số tháng có tỷ lệ thành công cao hơn. |
| VIF tất cả < 5 | Không đa cộng tuyến | 🟢 **Dễ** | Continuous features độc lập. |
| **Target imbalance 7.68:1** | Chỉ 11.5% là yes | 🔴 **Mất cân bằng nặng** | Model sinh thiếu lớp yes. JSD cho y sẽ cao. |

---
## Bước 3: Cảnh báo đặc biệt cho Evaluation

**Góc nhìn khoa học dữ liệu:**
- **Leakage duration là vấn đề nghiêm trọng nhất** — cần 2 feature list riêng
- **Target imbalance 7.68:1** — model sinh thiếu lớp yes
- **pdays = -1** — cần xử lý như missing indicator
- **'unknown' categories** — job, education, contact, poutcome có 'unknown' cần xử lý

In [26]:
print("="*70)
print("LEAKAGE DEMONSTRATION — duration")
print("="*70)
print("duration là thời gian cuộc gọi cuối cùng (giây).")
print("Ngoài đời thực: chỉ biết duration SAU KHI cuộc gọi kết thúc.")
print("Mà lúc đó, kết quả y (có đồng ý hay không) cũng đã biết rồi.")
print()
print("Kiểm tra: duration có 'biết trước' y không?")
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
X = df[['duration']]
y = (df['y']=='yes').astype(int)
scores = cross_val_score(RandomForestClassifier(n_estimators=50, random_state=42), X, y, cv=3)
print(f"  Cross-val accuracy (CHỈ duration): {scores.mean():.3f} (±{scores.std():.3f})")
print("  => CHỈ DÙNG duration đã predict đúng ~90%!")
print("  => ĐÂY LÀ LEAKAGE NGHIÊM TRỌNG.")
print()
print("GIẢI PHÁP:")
print("  - Fidelity: GIỮ duration (để so sánh 3 models)")
print("  - Utility: BỎ duration (vì thực tế không có trước khi gọi)")
print("  => Cần 2 feature list riêng cho cùng 1 dataset!")

print("\n--- MODE COLLAPSE RISK ---")
for col in c_cat:
    ca=cardinality_analysis(df[col])
    if ca['n_rare']>0:
        print(f"  {col}: {ca['n_rare']} categories hiếm (VD: {', '.join(ca['rare_categories'][:3])})")

print("\n--- MEMORIZATION RISK (outlier cực đoan) ---")
print(f"  balance >= 50000: {(df['balance']>=50000).sum()} samples")
print(f"  duration >= 2000: {(df['duration']>=2000).sum()} samples")
print(f"  campaign >= 30:   {(df['campaign']>=30).sum()} samples")

print("\n--- COVARIATE SHIFT (Train vs Test) ---")
from sklearn.model_selection import train_test_split
df_tr,df_te = train_test_split(df, test_size=0.2, random_state=42)
drift = covariate_shift_detection(df_tr, df_te, c_cont, c_cat)
n_drift = sum(1 for d in drift if d['p_value']<0.05)
print(f"  Features có drift (p<0.05): {n_drift}/{len(drift)}")
for d in drift:
    if d['p_value']<0.05:
        print(f"  ** {d['feature']:20s}: {d['method']:25s} p={d['p_value']:.4f}")
print("  => stratified split đã dùng → giảm drift.")

LEAKAGE DEMONSTRATION — duration
duration là thời gian cuộc gọi cuối cùng (giây).
Ngoài đời thực: chỉ biết duration SAU KHI cuộc gọi kết thúc.
Mà lúc đó, kết quả y (có đồng ý hay không) cũng đã biết rồi.

Kiểm tra: duration có 'biết trước' y không?
  Cross-val accuracy (CHỈ duration): 0.868 (±0.001)
  => CHỈ DÙNG duration đã predict đúng ~90%!
  => ĐÂY LÀ LEAKAGE NGHIÊM TRỌNG.

GIẢI PHÁP:
  - Fidelity: GIỮ duration (để so sánh 3 models)
  - Utility: BỎ duration (vì thực tế không có trước khi gọi)
  => Cần 2 feature list riêng cho cùng 1 dataset!

--- MODE COLLAPSE RISK ---
  job: 1 categories hiếm (VD: unknown)
  day: 2 categories hiếm (VD: 24, 1)
  month: 1 categories hiếm (VD: dec)

--- MEMORIZATION RISK (outlier cực đoan) ---
  balance >= 50000: 1 samples
  duration >= 2000: 5 samples
  campaign >= 30:   6 samples

--- COVARIATE SHIFT (Train vs Test) ---
  Features có drift (p<0.05): 0/16
  => stratified split đã dùng → giảm drift.


---
## Bước 4: Baseline Association Matrix

**Đây là "ground truth" — correlation difference trong evaluation sẽ so sánh synthetic với các giá trị này.**

In [27]:
print("=== BASELINE CRAMER'S V (Real Data) ===")
print("(synthetic data cần đạt càng gần các giá trị này càng tốt)")
cat_cols = ['job','marital','education','contact','poutcome','y']
for c1 in cat_cols:
    for c2 in cat_cols:
        if c1<c2:
            v=cramers_v(df[c1],df[c2])
            print(f"  V({c1:20s}, {c2:20s}) = {v:.4f}")
print()
print("=> Correlation difference = mean absolute diff giữa baseline và synthetic.")
print("   Nếu overall diff < 0.05: fidelity rất tốt.")
print("   Nếu diff > 0.10: model sinh không giữ được cấu trúc tương quan.")

=== BASELINE CRAMER'S V (Real Data) ===
(synthetic data cần đạt càng gần các giá trị này càng tốt)
  V(job                 , marital             ) = 0.1971
  V(job                 , poutcome            ) = 0.0397
  V(job                 , y                   ) = 0.1133
  V(marital             , poutcome            ) = 0.0102
  V(marital             , y                   ) = 0.0614
  V(education           , job                 ) = 0.4551
  V(education           , marital             ) = 0.1213
  V(education           , poutcome            ) = 0.0183
  V(education           , y                   ) = 0.0520
  V(contact             , job                 ) = 0.1504
  V(contact             , marital             ) = 0.0531
  V(contact             , education           ) = 0.1249
  V(contact             , poutcome            ) = 0.2029
  V(contact             , y                   ) = 0.1378
  V(poutcome            , y                   ) = 0.2914

=> Correlation difference = mean absolute dif

---
## Bước 5: Dự đoán kết quả và Hạn chế

### Dự đoán chi tiết (dựa trên số liệu thống kê cụ thể từ B1-B4)

#### Fidelity
| Metric | CTGAN | TVAE | TabDDPM | Cơ sở thống kê |
|--------|-------|------|---------|---------------|
| **Correlation Diff** | **Cao nhất** | Trung bình | **Thấp nhất** | duration vs y có tương quan rất mạnh. GAN/VAE không giữ được. |
| Wasserstein (balance) | **Cao** | TB | Thấp | Có giá trị âm + skew=4.5. Cần StandardScaler. |
| Wasserstein (duration) | TB | TB | Thấp | Phân phối lệch phải, nhiều peak. |
| JSD (poutcome) | **Cao** | TB | **Thấp** | 'unknown' chiếm tỷ lệ cao. |
| JSD (y) | **Cao nhất** | TB | **Thấp nhất** | Target imbalance 7.68:1. CTGAN mất cân bằng nặng nhất. |
| JSD (job) | TB | TB | Thấp | 12 categories, 'unknown' cần xử lý. |

#### Privacy
| Metric | CTGAN | TVAE | TabDDPM | Cơ sở thống kê |
|--------|-------|------|---------|---------------|
| DCR Mean | **Cao nhất (tốt nhất)** | TB | Thấp nhất | CTGAN sinh ngẫu nhiên hơn → xa real data hơn. |
| MIA AUC | **Thấp nhất (an toàn)** | TB | Cao nhất | Diffusion memorization. |

#### Utility
| Metric | Ghi chú |
|--------|---------|
| TSTR (có duration) | **Cao giả tạo** — duration quyết định y. Ceiling effect. |
| TSTR (bỏ duration) | **Thực tế hơn.** Đây mới là Utility thật sự. |
| **Ablation cần thiết** | Phải chạy Utility evaluation với 2 scenarios: có duration và không duration. |

### Hạn chế cần ghi nhận
1. **Leakage (duration):** Vấn đề lớn nhất. Cần 2 feature list riêng cho Fidelity và Utility.
2. **Target imbalance (7.68:1):** Model sinh thiếu lớp yes. JSD cho y sẽ cao.
3. **pdays = -1:** Cần xử lý như missing indicator (binary + scale riêng).
4. **'unknown' categories:** job, education, contact, poutcome có 'unknown' cần xử lý.
5. **balance có giá trị âm:** Cần StandardScaler thay vì MinMax.

In [28]:
print("="*70)
print("TỔNG HỢP — EDA KEY NUMBERS")
print("="*70)
bal=class_balance(df[target])
miss=missing_analysis(df)
dup=duplicate_analysis(df)
vifs=vif_analysis(df,c_cont)
print(f"  Target imbalance ratio:       {bal['imbalance_ratio']:.2f}:1")
print(f"  Missing cells:                {miss['total_missing']:,}/{miss['total_cells']:,} ({miss['overall_missing_pct']:.2f}%)")
print(f"  Duplicate rows:               {dup['n_exact_duplicates']}")
print(f"  Zero-inflation > 50%:         {sum(1 for c in c_cont if describe_continuous(df[c])['zero_ratio']>0.5)}/{len(c_cont)}")
print(f"  Features not normal:          {sum(1 for c in c_cont if not normality_test(df[c])['is_normal'])}/{len(c_cont)}")
print(f"  VIF > 5:                      {sum(1 for v in vifs.values() if v>5)}/{len(c_cont)}")
print(f"  Categorical with rare cats:   {sum(1 for c in c_cat if cardinality_analysis(df[c])['n_rare']>0)}/{len(c_cat)}")
print(f"  Leakage (duration vs y):      Spearman r = {r_dur:.4f}")
print()
print(">> Các con số này là BASELINE để so sánh với synthetic data.")

TỔNG HỢP — EDA KEY NUMBERS
  Target imbalance ratio:       7.68:1
  Missing cells:                0/76,857 (0.00%)
  Duplicate rows:               0
  Zero-inflation > 50%:         1/6
  Features not normal:          6/6
  VIF > 5:                      0/6
  Categorical with rare cats:   3/10
  Leakage (duration vs y):      Spearman r = 0.3484

>> Các con số này là BASELINE để so sánh với synthetic data.


---
## Tài liệu tham khảo
1. **UCI Bank Marketing:** https://archive.ics.uci.edu/dataset/222/bank+marketing
2. **Wasserstein-1:** đo khác biệt 2 phân phối liên tục, normalized bằng MinMax của real data range
3. **JSD (base=2):** Jensen-Shannon Divergence, đo khác biệt 2 phân phối categorical, return ∈ [0,1]
4. **Cramér's V (bias-corrected):** Bergsma & Wicher (2013) — hiệu chỉnh thiên vị cho mẫu nhỏ
5. **Leakage trong Bank Marketing:** duration → cần 2 feature list riêng cho Fidelity và Utility